---
# Методы статистической обработки информации
---
## Лабораторная работа 3 / Компонентный анализ / Вариант 15
---

---
## Подготовка
---

### Импорт библиотек

In [27]:
from MoSDP_data       import get_data
from MoSDP_operations import *

### Получение данных варианта

In [28]:
variant  = 15 # int(input("Введите номер варианта: "))
var_data = get_data(variant)

### Подготовка данных

In [29]:
Y = var_data[0]
X = var_data[1:]

k = len(X)
n = len(X[0])

---
## Задание 1
---

### Матрица нормированных значений

In [30]:
X_means = arithmetic_mean_deviations(X)
S       = root_mean_square_deviations(X)

Z = copy(X)
for i in range(len(X)):
    for j in range(len(X[0])):
        Z[i][j] = (Z[i][j] - X_means[i]) / S[i]

Z = transpose(Z)
matrix_output(Z)

[0.8446041777183667       | 0.6431302302653678       | -0.18827077328892822     | 0.18253424486556716      | -0.39725755984274863     ]
[0.2755700071369258       | -0.2675618133546759      | -0.5596734153849532      | 0.7755307846652638       | -0.25341524363987294     ]
[-1.0521763908864343      | 0.6431302302653678       | -0.38635218240680796     | 1.6057259403848394       | 1.479133251520133        ]
[-0.6728202771654758      | 1.2936245471368273       | 0.306932749505772        | 0.8178876803652424       | 0.6010510526100433       ]
[-2.190244732049316       | 0.6431302302653678       | 0.9506973291388824       | -0.2622131599842058      | 1.837665591906404        ]
[0.4652480639974061       | -0.7229078351646977      | 0.3812132779249771       | 1.6650255943648093       | -0.4337548639539256      ]
[-0.10378610658403481     | -0.33261124504182193     | 1.0249778575580875       | -0.6222467734340216      | 0.5044405417275153       ]
[-0.48314222030499543     | 0.9033279570139515  

### Матрица парных коэффициентов корреляции

In [31]:
R = matrix_by_a_number(matrix_multiplication(transpose(Z), Z), 1 / n)
matrix_output(R)

[1.0                      | -0.06874216425985699     | -0.016615678293707192    | 0.2193289370025392       | -0.9402329788072865      ]
[-0.06874216425985699     | 1.0                      | 0.4333027759567459       | -0.04467957562887467     | 0.14675271575071888      ]
[-0.016615678293707192    | 0.4333027759567459       | 1.0000000000000002       | -0.5084487611839537      | 0.037361580626634465     ]
[0.2193289370025392       | -0.04467957562887467     | -0.5084487611839537      | 1.0000000000000004       | -0.14710495223940964     ]
[-0.9402329788072865      | 0.14675271575071888      | 0.037361580626634465     | -0.14710495223940964     | 1.0                      ]


---
## Задание 2
---

### Функция "Характеристическое уравнение"

In [32]:
def characteristic_equation(matrix):
    P = []
    A = copy(matrix)

    for i in range(len(matrix)):
        if i == 0:
            P.append(tr(A))
        else:
            P.append(1 / (i + 1) * tr(A))

        B = copy(A)
        for j in range(len(A)):
            B[j][j] -= P[-1]

        A = matrix_multiplication(matrix, B)

    return P

### Функция "Формирование уравнения"

In [33]:
def draw(coefficients):
    equation = f"lambda**{k}"

    for i in range(k):
        if coefficients[i] >= 0:
            equation += " - "
        else:
            equation += " + "

        equation += f"{abs(coefficients[i])}"

        if i != k - 1:
            equation += f" * lambda**{k - i - 1}"

    return equation

### Функция "Поиск корней"

In [34]:
def roots(equation):
    q = 1
    l = []
    while len(l) != k:
        q   *= 10
        l    = []
        sign = "+" if eval(equation.replace("lambda", "0")) >= 0 else "-"

        for i in range(k * q):
            x        = i / q
            new_sign = "+" if eval(equation.replace("lambda", f"{x}")) >= 0 else "-"

            if sign != new_sign:
                sign = new_sign
                l.append(bisection_method(equation, x - 1 / q, x))

    return l

### Характеристическое уравнение

In [35]:
coefficients = characteristic_equation(R)
equation     = draw(coefficients)
components   = sorted(roots(equation), reverse=True)

print(f"Характеристическое уравнение: {equation.replace("**", "^")} = 0\n")
print(f"Корни характеристического уравнения: {components}")

Характеристическое уравнение: lambda^5 - 5.0 * lambda^4 + 8.57001537990404 * lambda^3 - 5.828855360148971 * lambda^2 + 1.329742080277674 * lambda^1 - 0.055218572156151446 = 0

Корни характеристического уравнения: [2.100261882153791, 1.5538874392161235, 0.959215601463438, 0.3337902784169273, 0.052844798750084035]


### Отбор главных компонент

In [36]:
first_component = []

sum_l = 0
for l in components:
    if sum_l / k < 0.6:
        sum_l += l
        first_component.append(l)
    else:
        break

print(f"Первые ({len(first_component)} << {k}) главные компоненты: {first_component}; вклад: {sum_l / k}")

Первые (2 << 5) главные компоненты: [2.100261882153791, 1.5538874392161235]; вклад: 0.7308298642739829


---
## Задание 3
---

### Функция "Собственный вектор"

In [37]:
def eigenvector(R, l):
    A = copy(R)

    complements = []
    for i in range(len(R)):
        A[i][i] -= l

    v = 0
    for i in range(len(R)):
        complements.append(complement(A, 1, i))
        v += complements[-1] ** 2

    v **= 0.5

    complements = [i / v for i in complements]

    return complements

### Матрица собственных векторов

In [46]:
U = []
for i in range(k):
    U.append(eigenvector(R, components[i]))

U = transpose(U)
matrix_output(U)

[-0.6079480478732704      | -0.3553022282289382      | 0.0194462794086681       | -0.045991704918704975    | -0.70828391411779        ]
[0.24561272723447725      | -0.40853283019043873     | 0.7550503134588095       | -0.4480441993312841      | 0.043940123681489716     ]
[0.2857609391182177       | -0.6538418093981605      | -0.0341735522429018      | 0.6988125568451373       | 0.03639715208061208      ]
[-0.34451463894495715     | 0.40102908148068295      | 0.6472253853451749       | 0.5437420316770305       | 0.07700157733184587      ]
[0.6080492648909479       | 0.34427869141415207      | 0.09722312098881182      | 0.11465937339596838      | -0.6993919980537859      ]


### Матрица факторных нагрузок

In [45]:
L = [[0 for _ in range(k)] for _ in range(k)]
for i in range(k):
    L[i][i] = components[i] ** 0.5

A = matrix_multiplication(U, L)
matrix_output(A)

[-0.8810553515973053      | -0.4429020690884617      | 0.019045599100971863     | -0.02657151709798009     | -0.16282028206789576     ]
[0.35594884876652266      | -0.5092566874230864      | 0.7394929008779759       | -0.25885568113262364     | 0.010100954136208986     ]
[0.4141327627719603       | -0.8150466482647375      | -0.033469424263456234    | 0.4037360614335886       | 0.008366976081356848     ]
[-0.4992802714111873      | 0.49990288785352877      | 0.6338896484106677       | 0.31414470755400453      | 0.017701119975963506     ]
[0.8812020381364092       | 0.42916067689883636      | 0.09521988997405938      | 0.06624397825694792      | -0.16077620870577725     ]


---
## Задание 4
---

### Матрица значений главных компонент

In [47]:
F = matrix_multiplication(Z, inverse_matrix(transpose(A)))
matrix_output(F)

[-0.4925055245197643      | -0.4037511064794108      | 0.6003418760702955       | -0.7007567264906589      | -1.239423655301867       ]
[-0.5619904387179838      | 0.48221078323231403      | 0.30607409398445945      | 0.18819870915344952      | 0.04195522815760555      ]
[0.7130796276856795       | 1.216871052349203        | 1.6963629967908274       | 0.9224644859741737       | -0.6586646825140847      ]
[0.6197604319223093       | 0.03594463861771488      | 1.5733914130158086       | 0.3106343034989275       | 0.8142059010494376       ]
[2.0486186881128643       | 0.33802678465090247      | 0.42829217051653845      | 0.9434439234240259       | 1.34303818316892         ]
[-0.8203237490393137      | 0.32021642894484215      | 0.4958813477823715       | 2.4656240292982914       | 0.3660909546589757       ]
[0.5488435551811675       | -0.45989824086967124     | -0.6553779529969082      | 1.0204527441207523       | -1.3246666999830115      ]
[0.6385885177306019       | -0.7824147618922278 

---
## Задание 5
---